##### Generalised Linear Model

In [1]:
#libraries
import pandas as pd 
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import os
import arviz as az

# Import functions
from cmdstanpy import CmdStanModel
from tensorflow_probability.substrates import numpy as tfp
tfd = tfp.distributions

# Create ./stan folder if does not exists
if not os.path.exists("./stan"):
    os.mkdir("./stan")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#get the data
data=pd.read_csv("/Users/sararedaelli/Desktop/transconjugacy efficacy PROJECT/data.csv")
print(data.head())
data_donors=pd.read_csv("/Users/sararedaelli/Desktop/transconjugacy efficacy PROJECT/data_donatori.csv")
print(data_donors.head())

#remember to drom TMTC and outliers!!!!   (TO DO)

#data cleaning 
data=data[data['Conta']!='TMTC']
print(data.head())
data_donors=data_donors[data_donors['Conta']!='TMTC']
print(data_donors.head())

  Idx Experiment  Idx Replica Diluizione Idx Replica Condizione Controllo  \
0              1                       1                      A        No   
1              1                       2                      A        No   
2              1                       3                      A        No   
3              1                       1                      A        No   
4              1                       2                      A        No   

   Diluition   IBU  DMSO  Temp Conta  
0         -1  6.25     0    28  TMTC  
1         -1  6.25     0    28  TMTC  
2         -1  6.25     0    28  TMTC  
3         -2  6.25     0    28    36  
4         -2  6.25     0    28    33  
  Idx Experiment  Idx Replica Diluizione  Diluition   IBU  DMSO  Temp  Conta
0              1                       1         -6  6.25     0    28     51
1              1                       2         -6  6.25     0    28     48
2              1                       3         -6  6.25     0    28   

In [3]:
#dimensions and colunms
print("data_trasconjugant shape:", data.shape)
print("data_trasconjugant columns:", data.columns.tolist())
print("data_donors shape:", data_donors.shape)
print("data_donors columns:", data_donors.columns.tolist())

data_trasconjugant shape: (378, 9)
data_trasconjugant columns: ['Idx Experiment', 'Idx Replica Diluizione', 'Idx Replica Condizione', 'Controllo', 'Diluition', 'IBU', 'DMSO', 'Temp', 'Conta']
data_donors shape: (72, 7)
data_donors columns: ['Idx Experiment', 'Idx Replica Diluizione', 'Diluition', 'IBU', 'DMSO', 'Temp', 'Conta']


In [4]:
#Rename columns
data.rename(columns={'Idx Replica Condizione': 'Idx Replica Experiment', 'Conta':'Count','Idx Replica Diluizione':'Idx Replica Diluition','Conta':'Count'}, inplace=True)
data_donors.rename(columns={'Idx Replica Diluizione':'Idx Replica Experiment','Conta':'Count'}, inplace=True)

#Modify values in some columns:
#Modify Idx Experiment such that experiments 5 and 6 have the same idx, experiments 7 and 8, experiment 5,6 has the same index as 5 and 6 and experiment 7,8 the same as 7 and 8
#but first convert all to string to avoid problems with mixed types
data['Idx Experiment'] = data['Idx Experiment'].astype(str)
data['Idx Experiment'] = data['Idx Experiment'].replace({'6': '5', '7':'6','8': '6', '9':'7','10':'8','5,6': '5', '7,8':'6'})
#now reconvert to numeric
data['Idx Experiment'] = pd.to_numeric(data['Idx Experiment'])

data_donors['Idx Experiment'] = data_donors['Idx Experiment'].astype(str)
data_donors['Idx Experiment'] = data_donors['Idx Experiment'].replace({'6': '5', '7':'6','8': '6', '9':'7','10':'8','5,6': '5', '7,8':'6'})
#now reconvert to numeric
data_donors['Idx Experiment'] = pd.to_numeric(data_donors['Idx Experiment'])

print(data['Idx Experiment'].unique())
print(data_donors['Idx Experiment'].unique())

#convert to numeric all the others columns I need 
#convert A;B;C in 1;2;3
data['Idx Replica Experiment'] = data['Idx Replica Experiment'].astype(str)
data['Idx Replica Experiment'] = data['Idx Replica Experiment'].replace({'A': '1', 'B':'2','C': '3'})
#now reconvert to numeric
data['Idx Replica Experiment'] = pd.to_numeric(data['Idx Experiment'])


data['Diluition'] = pd.to_numeric(data['Diluition'])
data['IBU'] = pd.to_numeric(data['IBU'])
data['DMSO'] = pd.to_numeric(data['DMSO'])
data['Temp'] = pd.to_numeric(data['Temp'])



[1 2 3 4 5 6 7 8]
[1 2 3 4 5 6 7 8]


In [6]:
#Extract the data we need for the Stan model 
#from trasconjugant dataset
Y=data.Count.values
N=len(Y)
idx_experiment=data['Idx Experiment'].values
I=len(np.unique(idx_experiment))
idx_experiment_replica=data['Idx Replica Experiment'].values
J=len(np.unique(idx_experiment_replica))
dil_trasc=data['Diluition'].values
rho_trasc = np.power(10.0, dil_trasc)


#from donor dataset
D=data_donors.Count.values
M=len(D)
idx_donor_experiment=data_donors['Idx Experiment'].values
dil_donor=data_donors['Diluition'].values
rho_donor=np.power(10.0, dil_donor)


#build design matrix X
IBU=data['IBU'].values
DMSO=data['DMSO'].values
Temp=data['Temp'].values
X=np.column_stack([np.ones_like(Y),IBU,DMSO,Temp])




##### Stan Model 

In [7]:
glm = """
data {
    int<lower=0> N; 
    int<lower=0> M; 
    int<lower=0> p; 
    int<lower=0> I;
    int<lower=0> J;
    array[N] int<lower=0> Y;
    array[M] int<lower=0> D;
    matrix[N, p] X;
    vector[N] rho_trasc;
    vector[M] rho_donor;
    array[N] int idx_experiment;
    array[N] int idx_experiment_replica;
    array[M] int idx_donor_experiment;
}

parameters {
    vector[p] beta;
    matrix[I, J] beta_random;
    vector<lower=0>[I] alpha;
    real<lower=0> sigma_beta;
    real<lower=0> tau;
    real mu_chi;
    real<lower=0> sigma_chi;
}


transformed parameters {
    vector[N] mu;
    for(i in 1:N) {
      mu[i] = alpha[idx_experiment[i]]* exp(X[i] * beta + beta_random[idx_experiment[i], idx_experiment_replica[i]]);
    }
}

model {   
    for (s in 1:N) {
        Y[s] ~ poisson(rho_trasc[s] * mu[s]);  

    }

    for (l in 1:M){
        D[l] ~ poisson(rho_donor[l] * alpha[idx_donor_experiment[l]]);
    }

    for (m in 1:I) {
        log(alpha[m])~normal(mu_chi, sigma_chi);

    }
    mu_chi ~ normal(0,5);
    sigma_chi ~ normal(0,1);  #it's halfNormal
    

    for (k in 1:p) {
        beta[k] ~ normal(0.0, tau);
        
    }

    tau ~ normal(0,1);  #HalfNormal
    
    for (i in 1:I) {    
        for (j in 1:J) {
            beta_random[i,j] ~ normal(0, sigma_beta);
        }
    }
    sigma_beta ~ inv_gamma(2, 1);
 
}

generated quantities {
  vector[N] log_lik;
  for (j in 1:N) {
    log_lik[j] = poisson_lpmf(Y[j] | rho_trasc[j]* mu[j]);
  }
}
"""


# Write stan model to file
stan_file = "./stan/glm.stan"
with open(stan_file, "w") as f:
    print(glm, file=f)

# Compile stan model
glm = CmdStanModel(stan_file=stan_file)

18:43:20 - cmdstanpy - INFO - compiling stan file /var/folders/17/4ypkwf497x10dw9zl6kjtsg80000gn/T/tmp3_ycvyh0/tmpw7v3n_xi.stan to exe file /Users/sararedaelli/Desktop/transconjugacy efficacy PROJECT/stan/glm


ValueError: Failed to compile Stan model '/Users/sararedaelli/Desktop/transconjugacy efficacy PROJECT/stan/glm.stan'. Console:

--- Translating Stan model to C++ code ---
bin/stanc --filename-in-msg=glm.stan --o=/var/folders/17/4ypkwf497x10dw9zl6kjtsg80000gn/T/tmp3_ycvyh0/tmpw7v3n_xi.hpp /var/folders/17/4ypkwf497x10dw9zl6kjtsg80000gn/T/tmp3_ycvyh0/tmpw7v3n_xi.stan
Syntax error in 'glm.stan', line 51, column 29, lexing error:
   -------------------------------------------------
    49:      }
    50:      mu_chi ~ normal(0,5);
    51:      sigma_chi ~ normal(0,1);  #it's halfNormal
                                       ^
    52:  
    53:  
   -------------------------------------------------

Invalid character found.
make: *** [/var/folders/17/4ypkwf497x10dw9zl6kjtsg80000gn/T/tmp3_ycvyh0/tmpw7v3n_xi.hpp] Error 1

Command ['make', 'STANCFLAGS+=--filename-in-msg=glm.stan', '/var/folders/17/4ypkwf497x10dw9zl6kjtsg80000gn/T/tmp3_ycvyh0/tmpw7v3n_xi']
	exited with code '2' No such file or d

In [ ]:
# Input data
glm_data = {
    "N": N,
    "M": M,
    "I":I,
    "J":J,
    "p": X.shape[1],
    "Y": Y,
    "D": D,
    "X": X,
    "rho_trasc": rho_trasc,
    "rho_donor": rho_donor,
    "idx_experiment": idx_experiment,
    "idx_experiment_replica": idx_experiment_replica,
    "idx_donor_experiment": idx_donor_experiment
}

# Sample
glm_fit = glm.sample(data=glm_data, chains=4, parallel_chains=4, 
                             iter_warmup=1000, iter_sampling=5000)

# Convert to arviz data type
poi_glm_data = az.from_cmdstanpy(glm_fit)

In [ ]:
#IDEE:
#una volta che faccio sampling:
#posterior vs prior predictive checks
#posterior predictive checks
#ESS
#compute posterior mean 
#compute credible intervals

#TO DO
#provo a runnare
#scegliere bene le prior 